In [1]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}

df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


A. Membaca dan Eksplorasi Awal

Data dibaca langsung dari HDFS, kemudian ditampilkan struktur, jumlah baris, dan 10 data pertama.

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

df.printSchema()

print("Jumlah baris:", df.count())

df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

B. Menangani Data Kosong

Kolom rating pada dataset memiliki beberapa nilai kosong. Pada bagian ini digunakan df.na.fill() untuk mengganti nilai kosong dengan nilai tertentu.

In [3]:
print("Jumlah data kosong pada rating:")

df.filter(df["rating"].isNull()).count()

Jumlah data kosong pada rating:


204

Kemudian data kosong ditangani dengan mengganti nilai rating yang kosong
menggunakan nilai rata-rata rating.

In [4]:
from pyspark.sql.functions import avg

rata_rating = df.select(avg("rating")).collect()[0][0]

df = df.na.fill({"rating": rata_rating})

print("Data kosong pada kolom rating sudah ditangani.")

Data kosong pada kolom rating sudah ditangani.


C. Transformasi Data

dibuat dua kolom baru, yaitu total_pendapatan dan tier_transaksi. total_pendapatan diperoleh dari perkalian unit_terjual dengan harga_satuan. Sedangkan tier_transaksi bernilai Besar jika total pendapatan lebih dari 500000 dan Kecil jika tidak.

In [5]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar")
    .otherwise("Kecil")
)

df.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------------------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|            rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------------------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|               4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|               5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|               3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman| 

D. Analisis dengan GroupBy
1. Kategori dengan Total Pendapatan Tertinggi

In [6]:
from pyspark.sql.functions import sum as spark_sum

pendapatan_kategori = df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(
    col("total_pendapatan").desc()
)

pendapatan_kategori.show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



2. Kota dengan Jumlah Transaksi Tier "Besar" Terbanyak

In [7]:
transaksi_besar = df.filter(
    col("tier_transaksi") == "Besar"
)

jumlah_per_kota = transaksi_besar.groupBy(
    "kota"
).count().orderBy(
    col("count").desc()
)

jumlah_per_kota.show()

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



3. Rata-rata Rating Berdasarkan Metode Pembayaran

In [8]:
rata_rating_pembayaran = df.groupBy(
    "metode_pembayaran"
).agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(
    col("rata_rata_rating").desc()
)

rata_rating_pembayaran.show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD| 4.167310656870009|
|    Transfer Bank|   4.1592349097265|
|         E-Wallet| 4.137728643216084|
|     Kartu Kredit|4.1179474608816475|
+-----------------+------------------+



E. Menyimpan Hasil ke HDFS

DataFrame hasil transformasi kemudian disimpan kembali ke HDFS dalam format CSV.

In [12]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi"

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Data berhasil disimpan ke HDFS.")

Data berhasil disimpan ke HDFS.


In [10]:
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi

Found 2 items
-rw-r--r--   3 aufaa supergroup          0 2026-09-17 08:16 /user/mahasiswa/tugas4/hasil_transaksi/_SUCCESS
-rw-r--r--   3 aufaa supergroup     100356 2026-09-17 08:16 /user/mahasiswa/tugas4/hasil_transaksi/part-00000-f0b79d83-958b-4bf5-bfc0-361e214baadd-c000.csv
